# Hopf CPU / GPU benchmark
Select a GPU runtime, then run these cells in order. The `colab_version` branch
must contain the updated files on GitHub before cloning. No source patching is needed.
The setup selects the assigned GPU architecture automatically.
Your original `Colab_version.ipynb` is preserved locally as a reference.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def run_logged(command, check=True):
    """Forward stdout and stderr into this notebook, including tracebacks."""
    print("$", " ".join(map(str, command)), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, command)
    return code


repo = Path("/content/CMODE")
if not repo.exists():
    run_logged(["git", "clone", "--branch", "colab_version", "--single-branch",
                "https://github.com/behzadtabari/CMODE.git", str(repo)])
branch = subprocess.check_output(["git", "-C", str(repo), "branch", "--show-current"], text=True).strip()
if branch != "colab_version":
    raise RuntimeError(f"Expected colab_version, found {branch}. Use a fresh runtime or switch the checkout.")
os.chdir(repo)
run_logged(["git", "pull", "--ff-only", "origin", "colab_version"])
run_logged(["git", "log", "-1", "--oneline"])
print("Repository:", repo, "Branch:", branch)


In [ ]:
# Build once. Both build logs and Python errors are displayed below.
import shutil
if shutil.which("nvcc") is None:
    raise RuntimeError("CUDA compiler missing. Select a GPU runtime with the CUDA toolkit.")
run_logged(["nvidia-smi"])
run_logged([sys.executable, "-m", "pip", "install", ".[test,examples]",
            "-Ccmake.define.ODELAB_ENABLE_CUDA=ON",
            "-Ccmake.define.CMAKE_CUDA_ARCHITECTURES=native"])
run_logged([sys.executable, "-c",
    "import odelab; print(odelab.__file__); assert odelab.cuda_available(), 'CUDA device unavailable'"])


In [ ]:
import csv
import html
import json
from datetime import datetime, timezone
from IPython.display import HTML, FileLink, display

# Change these values for your experiment.
counts = [1, 64, 256]
steps = 2000
h = 0.01
repeats = 3

# A separate directory per run prevents displaying stale results after a failure.
output = Path("build") / datetime.now(timezone.utc).strftime("benchmark_%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
json_path, csv_path = output / "results.json", output / "results.csv"
status = run_logged([
    sys.executable, "-u", "examples/hopfbi_gpu_cpu_benchmarking.py", "--require-cuda",
    "--counts", *map(str, counts), "--steps", str(steps), "--h", str(h),
    "--repeats", str(repeats), "--json", str(json_path), "--csv", str(csv_path),
], check=False)

if json_path.exists():
    results = json.loads(json_path.read_text())
    rows = results["rows"]
    columns = [
        ("batch", "Batch", "d"), ("backend", "Backend", "s"),
        ("median_ms", "Median time (ms)", ".3f"), ("min_ms", "Minimum time (ms)", ".3f"),
        ("speedup_vs_cpp", "Speedup vs C++", ".2f"),
        ("max_abs_error_ref", "Max abs. error vs reference", ".6e"),
        ("rmse_ref", "RMSE vs reference", ".6e"),
        ("max_abs_diff_cpp", "Max abs. difference vs C++", ".6e"),
    ]
    table = "<table><thead><tr>" + "".join(
        f"<th>{html.escape(title)}</th>" for _, title, _ in columns) + "</tr></thead><tbody>"
    for row in rows:
        table += "<tr>" + "".join(
            f"<td>{html.escape(format(row[key], fmt))}</td>" for key, _, fmt in columns) + "</tr>"
    display(HTML(table + "</tbody></table>"))
    print("Reference:", results["reference"])
    print("Errors cover all trajectories, time samples, and both state components.")
    print("Speedup = C++ median time / backend median time; above 1 means faster.")
    print("GPU times include allocations and transfers; reference generation is outside timings.")
    display(FileLink(str(csv_path)), FileLink(str(json_path)))
if status:
    raise RuntimeError("Benchmark failed. See the error output above; any completed results are shown.")
if not json_path.exists():
    raise RuntimeError("Benchmark produced no results file. Check the repository revision above.")


## Read the results
The table reports each backend separately for every batch size. The **reference
error** compares forward Euler with a tight-tolerance SciPy DOP853 numerical
solution; it is an estimate, not an error against a known exact solution.
The **difference vs C++** measures agreement between implementations and can be
near machine precision even when Euler's reference error is much larger.
All timings are median/minimum elapsed milliseconds over the chosen repetitions,
following a warm-up. Compilation, reference generation, and plotting are excluded.
The C++ CPU loop calls a Python RHS; CUDA evaluates it on-device. GPU elapsed time
includes transfers, allocation, and completion, rather than only kernel time.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for backend in dict.fromkeys(row["backend"] for row in rows):
    series = [row for row in rows if row["backend"] == backend]
    x = [row["batch"] for row in series]
    axes[0].plot(x, [row["median_ms"] for row in series], "o-", label=backend)
    axes[1].plot(x, [row["max_abs_error_ref"] for row in series], "o--", label=backend)
axes[0].set(xlabel="Batch size", ylabel="Median elapsed time (ms)", yscale="log", title="CPU / GPU timings")
axes[1].set(xlabel="Batch size", ylabel="Maximum absolute reference error", title="Euler error vs DOP853")
for ax in axes:
    ax.set_xscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.savefig(output / "benchmark.png", dpi=180)
plt.show()
display(FileLink(str(output / "benchmark.png")))


In [ ]:
run_logged([sys.executable, "-m", "pytest", "tests/python", "-q"])
run_logged([sys.executable, "examples/hopf_bifurcation.py",
                "--no-show", "--save", "build/hopf.png"])
from IPython.display import Image, display
display(Image(filename="build/hopf.png"))
